In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [3]:
import pandas as pd
import numpy as np

class BalancingMarketSettlement:
    def __init__(self, capacity_mwh: float, max_power_mw: float, penalty_rate: float = 1.2):
        self.capacity = capacity_mwh
        self.max_power = max_power_mw
        self.penalty_rate = penalty_rate

    def calculate_settlement(self, scheduled_mw: float, actual_activated_mw: float, clearing_price: float, capacity_price: float = 0.0, reserved_mw: float = 0.0) -> dict:
        """
        Berechnet die finanzielle Abrechnung (Settlement) für die Teilnahme am Regelenergiemarkt (aFRR/mFRR),
        einschließlich Leistungspreisen (Capacity Payments) und Bilanzkreisabweichungen (Imbalance Penalties).
        """
        # 1. Leistungserlös (Capacity Payment für bereitgehaltene Leistung)
        capacity_revenue = reserved_mw * capacity_price

        # 2. Arbeitserlös aus tatsächlicher Abruf-Energie (Energy Activation Revenue)
        energy_revenue = actual_activated_mw * clearing_price

        # 3. Bilanzierungsstrafe bei Abweichung vom Fahrplan (Imbalance Penalty)
        deviation = abs(scheduled_mw - actual_activated_mw)
        imbalance_penalty = deviation * clearing_price * self.penalty_rate

        # 4. Nettoumsatz
        net_revenue = capacity_revenue + energy_revenue - imbalance_penalty

        return {
            "Capacity_Revenue": capacity_revenue,
            "Energy_Revenue": energy_revenue,
            "Imbalance_Penalty": imbalance_penalty,
            "Net_Settlement": net_revenue
        }

    def simulate_time_series(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Simuliert das Settlement über eine Zeitreihe (DataFrame mit stündlichen/viertelstündlichen Werten).
        Erwartete Spalten: ['Scheduled_MW', 'Actual_MW', 'Clearing_Price', 'Capacity_Price', 'Reserved_MW']
        """
        results = []
        for _, row in df.iterrows():
            res = self.calculate_settlement(
                scheduled_mw=row['Scheduled_MW'],
                actual_activated_mw=row['Actual_MW'],
                clearing_price=row['Clearing_Price'],
                capacity_price=row.get('Capacity_Price', 0.0),
                reserved_mw=row.get('Reserved_MW', 0.0)
            )
            results.append(res)

        res_df = pd.DataFrame(results)
        return pd.concat([df.reset_index(drop=True), res_df], axis=1)
print("BalancingMarketSettlement Klasse erfolgreich geladen!")

BalancingMarketSettlement Klasse erfolgreich geladen!


In [4]:
# ایجاد یک تاریخچه زمانی فرضی (مثلا 24 ساعت)
np.random.seed(42)
hours = 24
timestamps = pd.date_range(start="2026-09-15 00:00", periods=hours, freq="h")

# داده‌های ساختگی بازار بالانسینگ (aFRR)
mock_data = pd.DataFrame({
    "Timestamp": timestamps,
    "Reserved_MW": 5.0, # ظرفیت رزرو شده برای بالانسینگ
    "Capacity_Price": np.random.uniform(10, 25, hours), # قیمت رزرو (EUR/MW)
    "Clearing_Price": np.random.uniform(50, 200, hours), # قیمت تسویه انرژی (EUR/MWh) - با احتمال اسپک بالا
    "Scheduled_MW": np.random.uniform(-2, 2, hours), # برنامه تجاری اعلام شده
    "Actual_MW": np.random.uniform(-2.5, 2.5, hours) # تزریق/تخلیه واقعی بر اساس فرمان شبکه
})

# تزریق چند قیمت اسپک شدید (مشابه بازار آلمان مثل روزی که بررسی کردیم)
mock_data.loc[19, "Clearing_Price"] = 740.0

# راه‌اندازی نمونه از کلاس (باتری مثلا 10 مگاوات ساعت با توان 5 مگاوات)
bess_settlement = BalancingMarketSettlement(capacity_mwh=10.0, max_power_mw=5.0, penalty_rate=1.2)

# اجرای شبیه‌سازی
results_df = bess_settlement.simulate_time_series(mock_data)

# نمایش 5 سطر اول نتایج
display(results_df.head())

,Timestamp,Reserved_MW,Capacity_Price,Clearing_Price,Scheduled_MW,Actual_MW,Capacity_Revenue,Energy_Revenue,Imbalance_Penalty,Net_Settlement
0,2026-09-15 00:00:00,5.0,15.618102,118.410498,0.186841,-2.472389,78.090509,-292.756861,377.856973,-592.523325
1,2026-09-15 01:00:00,5.0,24.260715,167.776394,-1.260582,1.577307,121.303573,264.634905,571.357005,-185.418527
2,2026-09-15 02:00:00,5.0,20.979909,79.951067,1.878339,1.034287,104.899546,82.692327,80.979410,106.612463
3,2026-09-15 03:00:00,5.0,18.979877,127.135166,1.100531,1.145036,94.899386,145.574321,6.789712,233.683996
4,2026-09-15 04:00:00,5.0,12.340280,138.862185,1.757996,1.356352,61.701398,188.345966,66.927802,183.119562


In [5]:
total_capacity_rev = results_df["Capacity_Revenue"].sum()
total_energy_rev = results_df["Energy_Revenue"].sum()
total_penalties = results_df["Imbalance_Penalty"].sum()
total_net = results_df["Net_Settlement"].sum()

print("--- FINANZIELLER GESAMTÜBERBLICK (24h) ---")
print(f"Leistungserlöse (Capacity Payments): {total_capacity_rev:,.2f} EUR")
print(f"Arbeitserlöse (Energy Revenue):      {total_energy_rev:,.2f} EUR")
print(f"Geldstrafen (Imbalance Penalties):  -{total_penalties:,.2f} EUR")
print(f"---------------------------------------------")
print(f"Nettoumsatz gesamt (Net Settlement):  {total_net:,.2f} EUR")

--- FINANZIELLER GESAMTÜBERBLICK (24h) ---
Leistungserlöse (Capacity Payments): 1,992.29 EUR
Arbeitserlöse (Energy Revenue):      906.57 EUR
Geldstrafen (Imbalance Penalties):  -4,601.11 EUR
---------------------------------------------
Nettoumsatz gesamt (Net Settlement):  -1,702.24 EUR


In [6]:
import pulp
import pandas as pd
import numpy as np

class BESS_CoOptimizer:
    def __init__(self, capacity_mwh: float, max_power_mw: float, efficiency: float = 0.9):
        # Initialize battery physical parameters
        self.capacity_mwh = capacity_mwh
        self.max_power_mw = max_power_mw
        self.efficiency = efficiency

    def optimize(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Co-optimizes Day-Ahead (DA) market and Balancing Capacity market (aFRR)
        using Mixed-Integer Linear Programming (MILP).
        """
        # 1. Initialize the optimization model (Maximization problem)
        model = pulp.LpProblem("BESS_CoOptimization", pulp.LpMaximize)

        time_steps = df.index.tolist()

        # 2. Define Decision Variables
        # Power charged from the grid (DA market)
        p_charge = pulp.LpVariable.dicts("Charge_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        # Power discharged to the grid (DA market)
        p_discharge = pulp.LpVariable.dicts("Discharge_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        # Capacity reserved for Balancing Market (aFRR Upward Regulation)
        cap_reserve = pulp.LpVariable.dicts("Reserve_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        # State of Charge (SoC) of the battery
        soc = pulp.LpVariable.dicts("SoC_MWh", time_steps, lowBound=0, upBound=self.capacity_mwh)

        # 3. Define the Objective Function: Maximize Total Profit
        # Profit = (DA Discharge Revenue) - (DA Charge Cost) + (Balancing Capacity Revenue)
        model += pulp.lpSum([
            p_discharge[t] * df.loc[t, 'DA_Price']
            - p_charge[t] * df.loc[t, 'DA_Price']
            + cap_reserve[t] * df.loc[t, 'Capacity_Price']
            for t in time_steps
        ]), "Total_Profit"

        # 4. Define Constraints
        for t in time_steps:
            # A. Power Limits & Reserve Allocation
            # The total discharged power plus reserved capacity cannot exceed the inverter's max power
            model += p_discharge[t] + cap_reserve[t] <= self.max_power_mw, f"Max_Power_Discharge_Limit_{t}"

            # For simplicity in this specific module, we prevent simultaneous charge and discharge
            # without binary variables by relying on market price spreads.
            model += p_charge[t] <= self.max_power_mw, f"Max_Power_Charge_Limit_{t}"

            # B. State of Charge (SoC) Tracking
            if t == time_steps[0]:
                # Initial SoC is assumed to be 50%
                model += soc[t] == (self.capacity_mwh * 0.5) + (p_charge[t] * self.efficiency) - (p_discharge[t] / self.efficiency), f"SoC_Tracking_{t}"
            else:
                # Subsequent SoC depends on previous SoC and current charge/discharge actions
                model += soc[t] == soc[t-1] + (p_charge[t] * self.efficiency) - (p_discharge[t] / self.efficiency), f"SoC_Tracking_{t}"

            # C. Energy Availability for Reserves (Crucial to avoid penalties!)
            # Battery must have enough energy to actually deliver the reserved capacity if activated for a full hour
            model += soc[t] >= (cap_reserve[t] / self.efficiency), f"Energy_Availability_For_Reserve_{t}"

        # 5. Solve the model
        model.solve(pulp.PULP_CBC_CMD(msg=0))

        # 6. Extract the results into the DataFrame
        df['Optimized_Charge_MW'] = [p_charge[t].varValue for t in time_steps]
        df['Optimized_Discharge_MW'] = [p_discharge[t].varValue for t in time_steps]
        df['Optimized_Reserve_MW'] = [cap_reserve[t].varValue for t in time_steps]
        df['Optimized_SoC_MWh'] = [soc[t].varValue for t in time_steps]

        return df

# --- Execution Block (Mock Data) ---
# Create 24 hours of mock market data
np.random.seed(10)
mock_market_data = pd.DataFrame({
    'DA_Price': np.random.uniform(20, 100, 24),        # Day-Ahead Energy Prices
    'Capacity_Price': np.random.uniform(5, 30, 24)     # Balancing Capacity Prices
})

# Add the extreme price spike we discussed (Hour 19)
mock_market_data.loc[19, 'DA_Price'] = 740.0

# Initialize and run the optimizer
# Assuming a 10 MWh battery with 5 MW max power
optimizer = BESS_CoOptimizer(capacity_mwh=10.0, max_power_mw=5.0)
optimized_results = optimizer.optimize(mock_market_data)

# Display the crucial hours around the price spike
display(optimized_results.loc[17:21])

,DA_Price,Capacity_Price,Optimized_Charge_MW,Optimized_Discharge_MW,Optimized_Reserve_MW,Optimized_SoC_MWh
17,43.350085,25.482175,5.00000,0.0,5.00,10.000000
18,93.421930,9.973688,0.00000,4.0,1.00,5.555556
19,740.000000,26.421258,0.00000,5.0,0.00,0.000000
20,63.403549,13.791316,1.17284,0.0,0.95,1.055556
21,31.373604,23.866192,5.00000,0.0,5.00,5.555556


In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming 'optimized_results' is the DataFrame from the MILP step
# We zoom in on the critical hours (e.g., 14:00 to 23:00)
df_plot = optimized_results.loc[14:23].copy()
hours = df_plot.index.tolist()

# Create a figure with secondary y-axis for market prices
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Add Battery Actions (Bar charts)
fig.add_trace(
    go.Bar(x=hours, y=df_plot['Optimized_Charge_MW'], name="Charge (MW)", marker_color='#00CC96', opacity=0.8),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=hours, y=-df_plot['Optimized_Discharge_MW'], name="Discharge (MW)", marker_color='#EF553B', opacity=0.8),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=hours, y=df_plot['Optimized_Reserve_MW'], name="aFRR Reserve (MW)", marker_color='#AB63FA', opacity=0.6),
    secondary_y=False,
)

# 2. Add Market Prices (Line charts on secondary axis)
fig.add_trace(
    go.Scatter(x=hours, y=df_plot['DA_Price'], name="Day-Ahead Price (€/MWh)", mode='lines+markers',
               line=dict(color='#FFA15A', width=3, dash='dot'), marker=dict(size=8, symbol='diamond')),
    secondary_y=True,
)

# 3. Add Annotations for the LinkedIn "Wow" factor
fig.add_annotation(
    x=19, y=740, xref="x", yref="y2",
    text="Price Spike: 740 €/MWh<br>AI Action: Max Discharge",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="#FFFFFF",
    ax=-60, ay=-40, font=dict(color="white", size=12), bgcolor="#EF553B"
)

# 4. Update Layout for Dark Theme and readability
fig.update_layout(
    title_text="BESS Co-Optimization: Day-Ahead vs. Balancing Market (aFRR)",
    title_font=dict(size=20, color='white'),
    template="plotly_dark",
    barmode='relative',
    hovermode="x unified",

    # --- رفع مشکل باکس هاور (کادری که با موس باز می‌شود) ---
    hoverlabel=dict(
        bgcolor="white",                  # پس‌زمینه سفید
        font=dict(color="blue", size=13), # متن آبی
        bordercolor="blue"                # حاشیه آبی
    ),

    # --- رفع مشکل باکس راهنما (Legend بالای نمودار) ---
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bgcolor="rgba(255, 255, 255, 0.95)", # پس‌زمینه سفید
        font=dict(color="blue", size=12)     # متن آبی
    ),
    margin=dict(l=40, r=40, t=80, b=40)
)

# Axis formatting
fig.update_yaxes(title_text="Battery Power (MW)", secondary_y=False, showgrid=False)
fig.update_yaxes(title_text="Market Price (€/MWh)", secondary_y=True, showgrid=True, gridcolor='#444')
fig.update_xaxes(title_text="Hour of Day", dtick=1)

fig.show()

In [12]:
!pip install -U kaleido pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
!pip install -U plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.8 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [2]:
!kaleido_get_chrome

/root/.local/share/choreographer/deps/chrome-linux64/chrome


In [4]:
!pip uninstall -y kaleido
!pip install kaleido==0.2.1.post1

Found existing installation: kaleido 1.4.0
Uninstalling kaleido-1.4.0:
  Successfully uninstalled kaleido-1.4.0
ERROR: Ignored the following yanked versions: 0.4.0, 0.4.1, 0.4.2
ERROR: Could not find a version that satisfies the requirement kaleido==0.2.1.post1 (from versions: 0.0.1rc3, 0.0.1rc4, 0.0.1rc5, 0.0.1rc6, 0.0.1rc8, 0.0.1rc9, 0.0.1, 0.0.2, 0.0.3, 0.0.3.post1, 0.1.0a2, 0.1.0a3, 0.1.0, 0.2.0rc1, 0.2.0, 0.2.1, 0.4.0rc1, 0.4.0rc2, 0.4.0rc3, 0.4.0rc4, 0.4.0rc5, 1.0.0rc0, 1.0.0rc11, 1.0.0rc13, 1.0.0rc15, 1.0.0, 1.1.0rc0, 1.1.0, 1.2.0, 1.3.0, 1.4.0)
ERROR: No matching distribution found for kaleido==0.2.1.post1


In [5]:
!pip install kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.3 MB/s eta 0:00:00


In [2]:
!pip install plotly==5.24.1 kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 36.3 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 7.0.0
    Uninstalling plotly-7.0.0:
      Successfully uninstalled plotly-7.0.0


In [1]:
import pandas as pd
import numpy as np
import pulp
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image

# ==========================================
# 1. BESS Optimizer Class (MILP)
# ==========================================
class BESS_CoOptimizer:
    def __init__(self, capacity_mwh: float, max_power_mw: float, efficiency: float = 0.9):
        self.capacity_mwh = capacity_mwh
        self.max_power_mw = max_power_mw
        self.efficiency = efficiency

    def optimize(self, df: pd.DataFrame) -> pd.DataFrame:
        model = pulp.LpProblem("BESS_CoOptimization", pulp.LpMaximize)
        time_steps = df.index.tolist()

        p_charge = pulp.LpVariable.dicts("Charge_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        p_discharge = pulp.LpVariable.dicts("Discharge_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        cap_reserve = pulp.LpVariable.dicts("Reserve_MW", time_steps, lowBound=0, upBound=self.max_power_mw)
        soc = pulp.LpVariable.dicts("SoC_MWh", time_steps, lowBound=0, upBound=self.capacity_mwh)

        model += pulp.lpSum([
            p_discharge[t] * df.loc[t, 'DA_Price']
            - p_charge[t] * df.loc[t, 'DA_Price']
            + cap_reserve[t] * df.loc[t, 'Capacity_Price']
            for t in time_steps
        ]), "Total_Profit"

        for t in time_steps:
            model += p_discharge[t] + cap_reserve[t] <= self.max_power_mw
            model += p_charge[t] <= self.max_power_mw
            if t == time_steps[0]:
                model += soc[t] == (self.capacity_mwh * 0.5) + (p_charge[t] * self.efficiency) - (p_discharge[t] / self.efficiency)
            else:
                model += soc[t] == soc[t-1] + (p_charge[t] * self.efficiency) - (p_discharge[t] / self.efficiency)
            model += soc[t] >= (cap_reserve[t] / self.efficiency)

        model.solve(pulp.PULP_CBC_CMD(msg=0))

        df['Optimized_Charge_MW'] = [p_charge[t].varValue for t in time_steps]
        df['Optimized_Discharge_MW'] = [p_discharge[t].varValue for t in time_steps]
        df['Optimized_Reserve_MW'] = [cap_reserve[t].varValue for t in time_steps]
        df['Optimized_SoC_MWh'] = [soc[t].varValue for t in time_steps]
        return df

# ==========================================
# 2. Generate Data & Run Optimization
# ==========================================
print("Running AI Optimization...")
np.random.seed(10)
mock_market_data = pd.DataFrame({
    'DA_Price': np.random.uniform(20, 100, 24),
    'Capacity_Price': np.random.uniform(5, 30, 24)
})
mock_market_data.loc[19, 'DA_Price'] = 740.0 # The Spike

optimizer = BESS_CoOptimizer(capacity_mwh=10.0, max_power_mw=5.0)
optimized_results = optimizer.optimize(mock_market_data)

# ==========================================
# 3. Create GIF Animation
# ==========================================
print("Generating GIF frames... Please wait.")
temp_dir = "temp_gif_frames"
os.makedirs(temp_dir, exist_ok=True)

df_plot = optimized_results.loc[14:23].copy()
all_hours = df_plot.index.tolist()
frames = []

y1_min, y1_max = -6, 12
y2_min, y2_max = 0, 850

for i in range(len(all_hours)):
    current_hours = all_hours[:i+1]
    current_df = df_plot.iloc[:i+1]

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Bar(x=current_hours, y=current_df['Optimized_Charge_MW'],
                         name="Charge (MW)", marker_color='#00CC96', opacity=0.8), secondary_y=False)
    fig.add_trace(go.Bar(x=current_hours, y=-current_df['Optimized_Discharge_MW'],
                         name="Discharge (MW)", marker_color='#EF553B', opacity=0.8), secondary_y=False)
    fig.add_trace(go.Bar(x=current_hours, y=current_df['Optimized_Reserve_MW'],
                         name="aFRR Reserve (MW)", marker_color='#AB63FA', opacity=0.6), secondary_y=False)

    fig.add_trace(go.Scatter(x=current_hours, y=current_df['DA_Price'],
                             name="Day-Ahead Price (€/MWh)", mode='lines+markers',
                             line=dict(color='#FFA15A', width=3, dash='dot'),
                             marker=dict(size=8, symbol='diamond')), secondary_y=True)

    if 19 in current_hours:
        fig.add_annotation(
            x=19, y=740, xref="x", yref="y2",
            text="Price Spike: 740 €/MWh<br>AI Action: Max Discharge",
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="#FFFFFF",
            ax=-60, ay=-40, font=dict(color="white", size=12), bgcolor="#EF553B"
        )

    fig.update_layout(
        title_text="BESS Co-Optimization: Day-Ahead vs. Balancing Market (aFRR)",
        title_font=dict(size=20, color='white'),
        template="plotly_dark",
        barmode='relative',
        showlegend=True,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            bgcolor="rgba(255, 255, 255, 0.95)", font=dict(color="blue", size=12)
        ),
        margin=dict(l=40, r=40, t=80, b=40),
        width=1000, height=600
    )

    fig.update_xaxes(title_text="Hour of Day", dtick=1, range=[all_hours[0]-0.5, all_hours[-1]+0.5])
    fig.update_yaxes(title_text="Battery Power (MW)", secondary_y=False, range=[y1_min, y1_max], showgrid=False)
    fig.update_yaxes(title_text="Market Price (€/MWh)", secondary_y=True, range=[y2_min, y2_max], showgrid=True, gridcolor='#444')

    img_path = f"{temp_dir}/frame_{i}.png"
    fig.write_image(img_path, scale=1.5)

    img = Image.open(img_path)
    frames.append(img)

    if i == len(all_hours) - 1:
        for _ in range(5):
            frames.append(img)

gif_path = 'bess_optimization_demo.gif'
frames[0].save(
    gif_path,
    format='GIF',
    append_images=frames[1:],
    save_all=True,
    duration=800,
    loop=0
)

print(f"✅ GIF successfully created and saved as: {gif_path}")

Running AI Optimization...
Generating GIF frames... Please wait.
✅ GIF successfully created and saved as: bess_optimization_demo.gif
